# Notebook 059: Streaming Deduplication Across Time Windows with Watermarks

**Module:** Module 06: Streaming & Near Real-Time Processing  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 11: Auto-loader & Video 23: Databricks Interview Questions  

---

## 1. Business Scenario & Real-World Context
An IoT gateway ingests sensor readings from connected vehicles over cellular networks. When vehicles enter tunnels or experience network reconnects, duplicate messages are re-sent. The pipeline must eliminate duplicate sensor IDs within a rolling 20-minute window.

## 2. Technical Requirement & Problem Statement
Implement streaming deduplication: apply `withWatermark()` paired with `dropDuplicates()` on primary keys and event timestamps. Explain how the watermark bounds the historical key state maintained by the streaming engine.

## 3. Spark & Databricks Architecture Notes
In Structured Streaming, calling `dropDuplicates(['key'])` without a watermark causes Spark to store every primary key ever seen in memory forever, eventually triggering an Executor OOM. Supplying a watermark allows Spark to evict keys older than the watermark threshold from the state store.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_059_Streaming_Deduplication_Across_Time_Windows_with_Watermarks")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Understanding Streaming Deduplication State Store Eviction
Illustrate how watermarking enables bounded key tracking in state stores.

In [ ]:
# Architectural Blueprint:
# stream.withWatermark("event_time", "20 minutes").dropDuplicates(["device_id", "event_time"])
# Keys older than (max_event_time - 20 minutes) are evicted from memory.
# New duplicates within the 20-minute window are safely identified and dropped!

print("Streaming deduplication memory bounds blueprint mapped.")

## 6. Simulating Sensor Stream with Duplicate Ingestion
Create dataset containing both unique and duplicate readings across time.

In [ ]:
from datetime import datetime, timedelta

base_ts = datetime(2023, 11, 20, 15, 0, 0)

telemetry_stream = [
    ("DEV-01", base_ts, 22.5),
    ("DEV-02", base_ts, 25.0),
    ("DEV-01", base_ts, 22.5), # Exact duplicate within window -> MUST DROP
    ("DEV-01", base_ts + timedelta(minutes=5), 23.0), # Valid new reading -> KEEP
    ("DEV-03", base_ts + timedelta(minutes=10), 19.5)
]

schema = "device_id STRING, event_time TIMESTAMP, temp DOUBLE"
df_stream_telemetry = spark.createDataFrame(telemetry_stream, schema)
df_stream_telemetry.show(truncate=False)

## 7. Executing Watermarked Streaming Deduplication
Apply watermark and dropDuplicates over device_id and event_time.

In [ ]:
# Deduplicate streaming data within a 20-minute watermark window
df_deduped = (
    df_stream_telemetry
    .withWatermark("event_time", "20 minutes")
    .dropDuplicates(["device_id", "event_time"])
)

print("Deduplicated Stream Output:")
df_deduped.orderBy("device_id", "event_time").show(truncate=False)

# Verification assertion
assert df_deduped.count() == 4, f"Expected 4 unique readings, found {df_deduped.count()}"
print("Streaming deduplication assertion verified successfully!")

## 9. Key Architectural Takeaways
- **Always combine `dropDuplicates([keys, timestamp_col])` with `withWatermark()` in streaming jobs.**
- **Omitting the watermark causes state store memory to grow monotonically until the cluster crashes.**
- **Duplicates arriving after the watermark has passed may be re-processed as new records; size the watermark to cover maximum expected network delay.**

## 10. Senior Data Engineering Interview Questions
1. What happens to the state store if you call `dropDuplicates(['id'])` without a watermark in Structured Streaming?
2. Why must the event-time column be included in `dropDuplicates` when using watermarks?
3. How does RocksDB state store handle state recovery during executor restarts?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
